# BGG GraphRAG QA

Semantic (vector) search over the board game knowledge graph — no SPARQL required.

**Pipeline:**
1. Load `bgg_main.ttl` + `fake_players.ttl` with rdflib
2. Convert each game / player to a natural-language text blurb
3. Embed with `sentence-transformers` (local, no API cost)
4. Store in two ChromaDB collections: `bgg_games` and `bgg_players`
5. At query time: embed the question, retrieve top-k blurbs, ask Claude

**First run:** expect ~2–3 min to build the index (21k enriched games).  
**Subsequent runs:** set `REBUILD_INDEX = False` in the config cell to skip embedding and go straight to queries.

In [ ]:
# cell: pip-install
%pip install -q chromadb sentence-transformers tqdm

In [ ]:
# cell: config
import os
from pathlib import Path

from rdflib import Graph, Namespace, RDF, RDFS, Literal
from sentence_transformers import SentenceTransformer
import chromadb
from tqdm import tqdm
import anthropic

BGG   = Namespace("https://raw.githubusercontent.com/susvej/bg_ontology/")
TRENJ = Namespace("https://vejdemo.se/boardgames/threnjen#")

# ── Configuration ────────────────────────────────────────────────────────────
EMBED_MODEL        = "all-MiniLM-L6-v2"   # fast 384-dim; swap all-mpnet-base-v2 for quality
CHROMA_PATH        = "./chroma_bgg"
GAMES_COLLECTION   = "bgg_games"
PLAYERS_COLLECTION = "bgg_players"
ANSWER_MODEL       = "claude-haiku-4-5-20251001"
TOP_K_GAMES        = 8     # game results per query
TOP_K_PLAYERS      = 3     # player results per query
REBUILD_INDEX      = False  # set False after first run to skip embedding

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
print("Config OK")

## Build index
Run once. Set `REBUILD_INDEX = False` in the config cell to skip on re-runs.

In [ ]:
# cell: load-ttl
if REBUILD_INDEX:
    print("Loading bgg_main.ttl ...")
    g = Graph()
    g.parse("bgg_main.ttl", format="turtle")
    print(f"  {len(g):,} triples")

    print("Loading fake_players.ttl ...")
    g.parse("fake_players.ttl", format="turtle")
    print(f"  {len(g):,} triples total")
    print("Done loading.")
else:
    print("Skipping TTL load (REBUILD_INDEX=False).")

In [ ]:
# cell: game-docs
def _lit(g, iri, pred):
    """Return string value of a datatype property, or None."""
    v = g.value(iri, pred)
    return str(v) if v is not None else None


def game_to_text(game_iri, g):
    name    = _lit(g, game_iri, BGG.hasName) or "Unknown"
    year    = _lit(g, game_iri, BGG.hasYearPublished)
    rating  = _lit(g, game_iri, BGG.hasRating)
    min_p   = _lit(g, game_iri, BGG.hasMinPlayers)
    max_p   = _lit(g, game_iri, BGG.hasMaxPlayers)
    best_p  = _lit(g, game_iri, BGG.hasBestNumPlayers)
    min_t   = _lit(g, game_iri, BGG.hasMinGameTime)
    max_t   = _lit(g, game_iri, BGG.hasMaxGameTime)
    age     = _lit(g, game_iri, BGG.hasMinRecAge)
    desc    = _lit(g, game_iri, BGG.hasDescription)
    enriched_val = g.value(game_iri, BGG.isFullyEnriched)
    enriched = bool(enriched_val) if enriched_val is not None else False

    header = name
    if year:
        header += f" ({year})"
    if rating:
        header += f" — Rating: {float(rating):.1f}/10"
    parts = [header]

    if min_p and max_p:
        pstr = f"Players: {min_p}–{max_p}"
        if best_p:
            pstr += f" (best: {best_p})"
        if min_t and max_t:
            pstr += f". Time: {min_t}–{max_t} min"
        elif min_t:
            pstr += f". Time: {min_t}+ min"
        if age:
            pstr += f". Age: {age}+"
        parts.append(pstr)

    if enriched:
        def labels(pred):
            return [str(g.value(v, RDFS.label) or "") for v in g.objects(game_iri, pred)
                    if g.value(v, RDFS.label)]
        cats    = labels(BGG.hasCategory)
        mechs   = labels(BGG.hasMechanic)
        themes  = labels(TRENJ.hasTheme)
        creators = labels(BGG.hasCreator)
        publishers = labels(BGG.hasPublisher)
        if cats:      parts.append("Categories: " + ", ".join(cats[:6]))
        if mechs:     parts.append("Mechanics: "  + ", ".join(mechs[:8]))
        if themes:    parts.append("Themes: "     + ", ".join(themes[:5]))
        if creators:  parts.append("Designers: "  + ", ".join(creators[:3]))
        if publishers: parts.append("Publisher: " + ", ".join(publishers[:2]))

    if desc:
        parts.append(desc[:400])

    return "\n".join(parts)


if REBUILD_INDEX:
    print("Generating game text blurbs ...")
    game_docs, game_ids, game_meta = [], [], []

    for game_iri in tqdm(list(g.subjects(RDF.type, BGG.Game))):
        gid = str(game_iri).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
        text = game_to_text(game_iri, g)
        rating_val = g.value(game_iri, BGG.hasRating)
        enriched_val = g.value(game_iri, BGG.isFullyEnriched)
        game_docs.append(text)
        game_ids.append(f"game_{gid}")
        game_meta.append({
            "bgg_id":   gid,
            "name":     str(g.value(game_iri, BGG.hasName) or ""),
            "rating":   float(str(rating_val)) if rating_val else 0.0,
            "enriched": bool(enriched_val) if enriched_val is not None else False,
        })

    print(f"  {len(game_docs):,} game documents ready")

In [ ]:
# cell: embed-games
if REBUILD_INDEX:
    print(f"Loading embedding model: {EMBED_MODEL}")
    model = SentenceTransformer(EMBED_MODEL)

    chroma = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        chroma.delete_collection(GAMES_COLLECTION)
        print("Cleared existing games collection")
    except Exception:
        pass
    games_col = chroma.create_collection(GAMES_COLLECTION)

    BATCH = 512
    print(f"Embedding {len(game_docs):,} games in batches of {BATCH} ...")
    for i in tqdm(range(0, len(game_docs), BATCH)):
        bd = game_docs[i:i+BATCH]
        bi = game_ids[i:i+BATCH]
        bm = game_meta[i:i+BATCH]
        be = model.encode(bd, show_progress_bar=False).tolist()
        games_col.add(documents=bd, embeddings=be, ids=bi, metadatas=bm)

    print(f"Games collection: {games_col.count():,} entries")
else:
    print("Loading embedding model (needed for queries) ...")
    model = SentenceTransformer(EMBED_MODEL)
    chroma = chromadb.PersistentClient(path=CHROMA_PATH)
    games_col = chroma.get_collection(GAMES_COLLECTION)
    print(f"Games collection loaded: {games_col.count():,} entries")

In [ ]:
# cell: players
def player_to_text(player_iri, g):
    name = str(g.value(player_iri, RDFS.label) or "Unknown Player")
    owned = list(g.objects(player_iri, BGG.hasOwnershipOf))
    owned_names = [str(g.value(gi, BGG.hasName) or gi) for gi in owned]

    opinions = []
    for op in g.subjects(BGG.hasOpinionHolder, player_iri):
        game_iri  = g.value(op, BGG.hasOpinionOf)
        gname     = str(g.value(game_iri, BGG.hasName) or "Unknown")
        rating    = g.value(op, BGG.hasPlayerRatingOpinion)
        load      = g.value(op, BGG.hasMentalLoad)
        load_str  = str(load).rsplit("/", 1)[-1] if load else None
        if rating:
            s = f"{gname} {int(float(str(rating)))}/10"
            if load_str:
                s += f" ({load_str})"
            opinions.append(s)

    parts = [f"{name} owns {len(owned)} games."]
    if owned_names:
        sample = owned_names[:8]
        tail   = f" and {len(owned_names)-8} more" if len(owned_names) > 8 else ""
        parts.append("Collection: " + ", ".join(sample) + tail)
    if opinions:
        parts.append("Ratings: " + "; ".join(opinions[:12]))
    return "\n".join(parts)


if REBUILD_INDEX:
    print("Generating player text blurbs ...")
    player_docs, player_ids, player_meta = [], [], []

    for p_iri in g.subjects(RDF.type, BGG.Player):
        pid  = str(p_iri).rsplit("/", 1)[-1].rsplit("#", 1)[-1]
        text = player_to_text(p_iri, g)
        player_docs.append(text)
        player_ids.append(f"player_{pid}")
        player_meta.append({"name": str(g.value(p_iri, RDFS.label) or "")})

    print(f"  {len(player_docs)} player documents ready")

    try:
        chroma.delete_collection(PLAYERS_COLLECTION)
    except Exception:
        pass
    players_col = chroma.create_collection(PLAYERS_COLLECTION)
    embs = model.encode(player_docs).tolist()
    players_col.add(documents=player_docs, embeddings=embs,
                    ids=player_ids, metadatas=player_meta)
    print(f"Players collection: {players_col.count()} entries")
else:
    players_col = chroma.get_collection(PLAYERS_COLLECTION)
    print(f"Players collection loaded: {players_col.count()} entries")

## Query
Run cells below any time — index must be built (or loaded) first.

In [ ]:
# cell: query-fn  ← RE-RUN THIS CELL whenever you change ask() or SYSTEM
import re
from IPython.display import display, Markdown

SYSTEM = """\
You are a helpful assistant answering questions about board games.
You are given context retrieved from a board game database: game descriptions
and player collection summaries.
Answer the question based on the provided context.
Be concise and direct. If the context does not contain enough information, say so.
Note: this is a semantic search system — results are ranked by similarity,
not sorted by rating or count, so avoid claiming exact rankings unless clearly
stated in the context. Where possible, put "I don't know" rather than guessing. 
Put answers in tables, or lists where appropriate.
"""

_PLAYER_KEYWORDS  = {"player", "owns", "own", "collection", "who", "opinion", "rated", "rating"}
_RANKING_KEYWORDS = {"top", "highest-rated", "highest rated", "best rated",
                     "most popular", "greatest", "all time", "overall best"}

_DESIGNER_PATTERNS = [
    re.compile(r"what games did (.+?) design", re.IGNORECASE),
    re.compile(r"games designed by (.+?)[\?.]?\s*$", re.IGNORECASE),
    re.compile(r"games by (.+?)[\?.]?\s*$", re.IGNORECASE),
]

def _is_ranking_query(question: str) -> bool:
    return any(kw in question.lower() for kw in _RANKING_KEYWORDS)

def _extract_designer(question: str):
    for pat in _DESIGNER_PATTERNS:
        m = pat.search(question.strip())
        if m:
            return m.group(1).strip().rstrip("?.")
    return None


def ask(question: str,
        top_k_games: int = TOP_K_GAMES,
        top_k_players: int = TOP_K_PLAYERS,
        question_id: str = None) -> str:
    q_emb = model.encode([question])[0].tolist()

    # --- Retrieve games ---
    designer = _extract_designer(question)

    if designer:
        # Exact text match for designer name → returns ALL their games
        print(f"[designer mode: '{designer}']")
        results = games_col.get(
            where_document={"$contains": designer},
            include=["documents", "metadatas"],
        )
        n = len(results["documents"])
        print(f"  Found {n} games")
        if n > 0:
            game_blurbs = results["documents"][:20]
        else:
            # Misspelling / no exact match → fall back to semantic search
            print("  No exact match — falling back to semantic search")
            g_res = games_col.query(query_embeddings=[q_emb], n_results=top_k_games)
            game_blurbs = g_res["documents"][0]

    elif _is_ranking_query(question):
        print("[ranking mode]")
        results = games_col.get(
            where={"rating": {"$gte": 7.5}},
            include=["documents", "metadatas"],
        )
        print(f"  games_col.get() returned {len(results['documents'])} docs")
        paired = sorted(
            zip(results["documents"], results["metadatas"]),
            key=lambda x: x[1].get("rating", 0),
            reverse=True,
        )
        game_blurbs = [doc for doc, _ in paired[:top_k_games]]
        if game_blurbs:
            top_meta = [m for _, m in paired[:top_k_games]]
            print(f"  Top game: {top_meta[0].get('name')} ({top_meta[0].get('rating')})")

    else:
        g_res = games_col.query(query_embeddings=[q_emb], n_results=top_k_games)
        game_blurbs = g_res["documents"][0]

    # --- Retrieve players if question is player-related ---
    q_lower = question.lower()
    include_players = any(kw in q_lower for kw in _PLAYER_KEYWORDS)
    player_blurbs = []
    if include_players:
        p_res = players_col.query(query_embeddings=[q_emb], n_results=top_k_players)
        player_blurbs = p_res["documents"][0]

    # --- Build context block ---
    ctx_parts = ["## Games\n"]
    for blurb in game_blurbs:
        ctx_parts.append(blurb)
        ctx_parts.append("---")
    if player_blurbs:
        ctx_parts.append("\n## Players\n")
        for blurb in player_blurbs:
            ctx_parts.append(blurb)
            ctx_parts.append("---")
    context = "\n".join(ctx_parts)

    response = client.messages.create(
        model=ANSWER_MODEL,
        max_tokens=1024,
        system=SYSTEM,
        messages=[{"role": "user",
                   "content": f"Context:\n{context}\n\nQuestion: {question}"}],
    )
    answer = response.content[0].text.strip()
    print(f"Q: {question}\n")
    display(Markdown(answer))
    print()
    if question_id:
        _log_qa(question_id, question, answer, "graphrag")
    if question_id:
        _log_qa(question_id, question, answer, "graphrag")
    return answer


print("ask() ready")

In [ ]:
# cell: diagnostics  ← run this to debug, delete when done
# 1. Does where_document find Rosenberg at all?
r1 = games_col.get(where_document={"$contains": "Uwe Rosenberg"}, include=["documents"])
print(f"where_document '$contains Uwe Rosenberg': {len(r1['documents'])} results")
if r1["documents"]:
    print("First match:\n", r1["documents"][0][:300])

print()

# 2. Does the index have ANY documents with "Designers:" in them?
r2 = games_col.get(where_document={"$contains": "Designers:"}, include=["documents"], limit=3)
print(f"where_document '$contains Designers:': {len(r2['documents'])} results (showing up to 3)")
for doc in r2["documents"]:
    print(" ---")
    print(doc[:200])

print()

# 3. Print a sample document so we can see its structure
r3 = games_col.get(include=["documents"], limit=1)
print("Sample document from index:")
print(r3["documents"][0])

In [22]:
ask("What are the top 10 highest-rated board games of all time?")

Q: What are the top 10 highest-rated board games of all time?

I don't have enough context to answer this question. The provided context only
includes 8 games from a board game database, which is not a comprehensive list
of "all time" board games, nor are they ranked by rating to determine a top 10.
From the games shown in the context, here are the highest-rated ones:  | Game |
Rating | |------|--------| | The Top 10 Game | 4.8/10 | | The Best of Board
Wargaming | 7.2/10 | | Kick-Ass: The Board Game | 7.3/10 | | European Union: The
Board Game | 6.7/10 | | Beowulf: The Movie Board Game | 6.3/10 | | Memory: The
Board Game | 6.3/10 | | Wolsung: The Boardgame | 5.4/10 | | Twilight: The Board
Game | 3.3/10 |  To get an accurate answer about the top 10 highest-rated games
of all time, you would need access to a more comprehensive board game database.



'I don\'t have enough context to answer this question. The provided context only includes 8 games from a board game database, which is not a comprehensive list of "all time" board games, nor are they ranked by rating to determine a top 10.\n\nFrom the games shown in the context, here are the highest-rated ones:\n\n| Game | Rating |\n|------|--------|\n| The Top 10 Game | 4.8/10 |\n| The Best of Board Wargaming | 7.2/10 |\n| Kick-Ass: The Board Game | 7.3/10 |\n| European Union: The Board Game | 6.7/10 |\n| Beowulf: The Movie Board Game | 6.3/10 |\n| Memory: The Board Game | 6.3/10 |\n| Wolsung: The Boardgame | 5.4/10 |\n| Twilight: The Board Game | 3.3/10 |\n\nTo get an accurate answer about the top 10 highest-rated games of all time, you would need access to a more comprehensive board game database.'

In [9]:
ask("Which games use the Worker Placement mechanic and have a rating above 8?")

Q: Which games use the Worker Placement mechanic and have a rating above 8?

Based on the context provided, there are **no games that use the Worker
Placement mechanic and have a rating above 8**.  Here are the Worker Placement
games in the database with their ratings: - Ground Floor (2012) — 7.0/10 -
Spyrium (2013) — 7.0/10 - Titans of Industry (2013) — 6.5/10 - Red Outpost
(2019) — 6.9/10 - Slate (2011) — 5.7/10 - Worker Placement (2014) — 5.4/10 -
Akua (2016) — 5.9/10  The highest-rated Worker Placement games are Ground Floor
and Spyrium, both at 7.0/10, which falls short of the 8+ threshold.



'Based on the context provided, there are **no games that use the Worker Placement mechanic and have a rating above 8**.\n\nHere are the Worker Placement games in the database with their ratings:\n- Ground Floor (2012) — 7.0/10\n- Spyrium (2013) — 7.0/10\n- Titans of Industry (2013) — 6.5/10\n- Red Outpost (2019) — 6.9/10\n- Slate (2011) — 5.7/10\n- Worker Placement (2014) — 5.4/10\n- Akua (2016) — 5.9/10\n\nThe highest-rated Worker Placement games are Ground Floor and Spyrium, both at 7.0/10, which falls short of the 8+ threshold.'

In [10]:
ask("What games can be played by exactly 2 players and take less than 30 minutes?")

Q: What games can be played by exactly 2 players and take less than 30 minutes?

Based on the provided context, I cannot identify any games that meet both
criteria of being playable by exactly 2 players AND taking less than 30 minutes.
Here's why: - **One-hour Wargames** supports 1-2 players but takes 60 minutes -
**Vector** supports 2-4 players but takes 60 minutes - All the other games
either don't support 2 players or take exactly 30 minutes (not less than 30)
The context does not contain enough information to answer this question fully,
as the game descriptions provided don't include games with play times under 30
minutes that accommodate exactly 2 players.



"Based on the provided context, I cannot identify any games that meet both criteria of being playable by exactly 2 players AND taking less than 30 minutes.\n\nHere's why:\n- **One-hour Wargames** supports 1-2 players but takes 60 minutes\n- **Vector** supports 2-4 players but takes 60 minutes\n- All the other games either don't support 2 players or take exactly 30 minutes (not less than 30)\n\nThe context does not contain enough information to answer this question fully, as the game descriptions provided don't include games with play times under 30 minutes that accommodate exactly 2 players."

In [11]:
ask("What games did Uwe Rosenberg design?")

Q: What games did Uwe Rosenberg design?

Based on the context provided, I don't see any games designed by Uwe Rosenberg
in the list. The designers mentioned in the database excerpt are:  - Thomas
Jansen - James Mathe - Michael Rinella - Rob Beyma - Benjamin Schwer - Klaus
Teuber - Peter Neugebauer - Philippe des Pallières  To find Uwe Rosenberg's
designs, you would need to search the full board game database.



"Based on the context provided, I don't see any games designed by Uwe Rosenberg in the list. The designers mentioned in the database excerpt are:\n\n- Thomas Jansen\n- James Mathe\n- Michael Rinella\n- Rob Beyma\n- Benjamin Schwer\n- Klaus Teuber\n- Peter Neugebauer\n- Philippe des Pallières\n\nTo find Uwe Rosenberg's designs, you would need to search the full board game database."

In [12]:
ask("What games did Uwe Rossenberg design?")
### Intentional misspelling of name

Q: What games did Uwe Rossenberg design?

Based on the context provided, I cannot find any games designed by Uwe Rosenberg
in the list of games shown. The designers mentioned in the context are: Hartmut
Witt, François-Stanislas Thomas, Chris Gosselin, Chris Kreuter, Joseph Miranda,
Gary Graber, Bernd Eisenstein, Harald Bilz, and Steve Finn.  To find games
designed by Uwe Rosenberg, you would need to search the board game database
directly or provide additional context containing his designs.



'Based on the context provided, I cannot find any games designed by Uwe Rosenberg in the list of games shown. The designers mentioned in the context are: Hartmut Witt, François-Stanislas Thomas, Chris Gosselin, Chris Kreuter, Joseph Miranda, Gary Graber, Bernd Eisenstein, Harald Bilz, and Steve Finn.\n\nTo find games designed by Uwe Rosenberg, you would need to search the board game database directly or provide additional context containing his designs.'

In [13]:
ask("Which categories have the most games?")

Q: Which categories have the most games?

Based on the provided context, **Word Game** and **Card Game** are tied for the
most games, each appearing in 3 games:  **Word Game (3 games):** - Category Game
(1994) - Scattergories: The Card Game (2008) - Scattergories Categories (2010)
**Card Game (3 games):** - Scattergories: The Card Game (2008) - Miscellaneous
Game Compilation - SuDoku: The Card Game (2006) - Unmatched: Jurassic Park –
InGen vs Raptors (2020)  Note: If counting Unmatched as well, Card Game would
have 4 games in this dataset.



'Based on the provided context, **Word Game** and **Card Game** are tied for the most games, each appearing in 3 games:\n\n**Word Game (3 games):**\n- Category Game (1994)\n- Scattergories: The Card Game (2008)\n- Scattergories Categories (2010)\n\n**Card Game (3 games):**\n- Scattergories: The Card Game (2008)\n- Miscellaneous Game Compilation\n- SuDoku: The Card Game (2006)\n- Unmatched: Jurassic Park – InGen vs Raptors (2020)\n\nNote: If counting Unmatched as well, Card Game would have 4 games in this dataset.'

In [14]:
ask("What games are in the Fantasy category with a geek rating above 7.5?")

Q: What games are in the Fantasy category with a geek rating above 7.5?

Based on the provided context, there are **no games in the Fantasy category with
a rating above 7.5**.  The Fantasy-themed games in the database are: - Diskwars:
Legend of the Five Rings (5.3/10) - Dungeon Crawler (6.7/10) - Dungeoneer:
Dragons of the Forsaken Desert (6.3/10)  All of these fall below the 7.5 rating
threshold.



'Based on the provided context, there are **no games in the Fantasy category with a rating above 7.5**.\n\nThe Fantasy-themed games in the database are:\n- Diskwars: Legend of the Five Rings (5.3/10)\n- Dungeon Crawler (6.7/10)\n- Dungeoneer: Dragons of the Forsaken Desert (6.3/10)\n\nAll of these fall below the 7.5 rating threshold.'

In [15]:
ask("Which fake players gave a rating of 10 to any game, and what games did they rate?")

Q: Which fake players gave a rating of 10 to any game, and what games did they rate?

Based on the context provided, **no players gave a rating of 10 to any game**.
The highest ratings in the player collection data are: - Klaus Pérez: 9/10
(Grabolo, Battle of the Bards) - Kwame Rodríguez: 8/10 (Legends of Draxia,
Piratissimo, The Beer Game, 1809: Napoleon's Danube Campaign) - Maja Brown: 9/10
(Champagne: The Game, Justice League Strategy Game)  None of the players rated
any game at 10/10.



"Based on the context provided, **no players gave a rating of 10 to any game**. \n\nThe highest ratings in the player collection data are:\n- Klaus Pérez: 9/10 (Grabolo, Battle of the Bards)\n- Kwame Rodríguez: 8/10 (Legends of Draxia, Piratissimo, The Beer Game, 1809: Napoleon's Danube Campaign)\n- Maja Brown: 9/10 (Champagne: The Game, Justice League Strategy Game)\n\nNone of the players rated any game at 10/10."

In [16]:
# Change this and run
ask("What are the best cooperative games?")

Q: What are the best cooperative games?

Based on the context provided, here are the cooperative games ranked by rating:
1. **Project: ELITE** (7.6/10) — A real-time cooperative dice rolling game for
1–4 players where you fight aliens. Features grid movement and variable player
powers.  2. **Mint Cooperative** (6.0/10) — A portable, quick-playing coop game
(15–30 min) for 1–4 players focused on teamwork to defeat a villain.  3.
**Emergence: A Game of Teamwork and Deception** (6.1/10) — A cooperative game
for 3–6 players that blends teamwork with deception mechanics.  4. **Professor
Pünschge** (6.1/10) — A 2–7 player cooperative game with deduction elements.
**Project: ELITE** stands out as the highest-rated cooperative option, combining
real-time gameplay with sci-fi themes. However, if you prefer something lighter
and more portable, **Mint Cooperative** is a good introductory option. For a
more complex experience with social deduction elements, **Emergence** might
appeal to you.



'Based on the context provided, here are the cooperative games ranked by rating:\n\n1. **Project: ELITE** (7.6/10) — A real-time cooperative dice rolling game for 1–4 players where you fight aliens. Features grid movement and variable player powers.\n\n2. **Mint Cooperative** (6.0/10) — A portable, quick-playing coop game (15–30 min) for 1–4 players focused on teamwork to defeat a villain.\n\n3. **Emergence: A Game of Teamwork and Deception** (6.1/10) — A cooperative game for 3–6 players that blends teamwork with deception mechanics.\n\n4. **Professor Pünschge** (6.1/10) — A 2–7 player cooperative game with deduction elements.\n\n**Project: ELITE** stands out as the highest-rated cooperative option, combining real-time gameplay with sci-fi themes. However, if you prefer something lighter and more portable, **Mint Cooperative** is a good introductory option. For a more complex experience with social deduction elements, **Emergence** might appeal to you.'

In [17]:
# Change this and run
ask("I like the game Set and the game Prosperitea. What other games with similar mechanics or categories might I like?")
### This is a great test to see if the agent will hallucinate when it doesn't have data. It does not!

Q: I like the game Set and the game Prosperitea. What other games with similar mechanics or categories might I like?

I don't have information about "Prosperitea" in the provided context, so I can't
make a direct comparison based on both games you mentioned.  However, based on
**Set** (a game known for pattern recognition and set collection), I can see one
game in the context with similar mechanics:  **Specific** (2016) — Rating:
5.9/10 - **Mechanics**: Pattern Recognition, Set Collection (along with Dice
Rolling and Modular Board) - **Players**: 2–10 - **Time**: 20 minutes - **Age**:
6+  This game shares the pattern recognition and set collection mechanics that
are core to Set.  If you could provide more details about what you enjoy about
Prosperitea (its mechanics, theme, or category), I could potentially suggest
other games from this database that might appeal to you.



'I don\'t have information about "Prosperitea" in the provided context, so I can\'t make a direct comparison based on both games you mentioned.\n\nHowever, based on **Set** (a game known for pattern recognition and set collection), I can see one game in the context with similar mechanics:\n\n**Specific** (2016) — Rating: 5.9/10\n- **Mechanics**: Pattern Recognition, Set Collection (along with Dice Rolling and Modular Board)\n- **Players**: 2–10\n- **Time**: 20 minutes\n- **Age**: 6+\n\nThis game shares the pattern recognition and set collection mechanics that are core to Set.\n\nIf you could provide more details about what you enjoy about Prosperitea (its mechanics, theme, or category), I could potentially suggest other games from this database that might appeal to you.'

In [18]:
ask("I like the game Set and the game Ricochet Robots. What other games with similar mechanics or categories might I like?")
### this breaks! Fun! figure out why and fix it! (hint: check the mechanics and categories of these games in the data, and see how the query is structured to find similar games)

Q: I like the game Set and the game Ricochet Robots. What other games with similar mechanics or categories might I like?

# Recommendations Based on Your Interests  Based on your enjoyment of **Set**
(pattern recognition) and **Ricochet Robots** (spatial puzzles with grid
movement), here are relevant options from the database:  **Best Match:** -
**Ricochet (1996)** — Uses pattern recognition and real-time puzzle solving like
Set, but in a compact card game format (10 min). Rating: 5.7/10  **Also Worth
Considering:** - **VOLT: Robot Battle Arena (2014)** — Combines grid movement
and programmed movement (like Ricochet Robots' puzzle-solving approach) with
action selection mechanics. 30 minutes, Rating: 6.9/10  **Note:** The database
has limited overlap with Set's core mechanic (pattern/set recognition). Most
other robot-themed games focus on combat, dice rolling, or campaign mechanics
rather than the abstract puzzle-solving you seem to enjoy.   If you're looking
for more games specifical

"# Recommendations Based on Your Interests\n\nBased on your enjoyment of **Set** (pattern recognition) and **Ricochet Robots** (spatial puzzles with grid movement), here are relevant options from the database:\n\n**Best Match:**\n- **Ricochet (1996)** — Uses pattern recognition and real-time puzzle solving like Set, but in a compact card game format (10 min). Rating: 5.7/10\n\n**Also Worth Considering:**\n- **VOLT: Robot Battle Arena (2014)** — Combines grid movement and programmed movement (like Ricochet Robots' puzzle-solving approach) with action selection mechanics. 30 minutes, Rating: 6.9/10\n\n**Note:** The database has limited overlap with Set's core mechanic (pattern/set recognition). Most other robot-themed games focus on combat, dice rolling, or campaign mechanics rather than the abstract puzzle-solving you seem to enjoy. \n\nIf you're looking for more games specifically matching Set's pattern recognition gameplay, you may need to explore beyond this robot-focused collection.

In [19]:
ask("I want an 8 player game that is not a party game. What are the best options?")

Q: I want an 8 player game that is not a party game. What are the best options?

Based on the provided context, I don't have enough information to answer your
question. The database contains party games and player collections, but it
doesn't include a comprehensive list of non-party games with 8-player support.
The context shows several games that support 8 players, but most are categorized
as party games: - **Spot it! Party** (2–8 players) - Party/Card Game -
**Hollywood Game Night Party Game** (4–8 players) - Party Game  While the player
collections mention some non-party games (like Terra Mystica, Euphrates &
Tigris, Advanced Dungeons & Dragons Battlesystem), none of these are explicitly
noted as supporting exactly 8 players in the provided information.  To get a
better recommendation, I would need access to a more comprehensive game database
that includes player counts and game categories for non-party titles.



"Based on the provided context, I don't have enough information to answer your question. The database contains party games and player collections, but it doesn't include a comprehensive list of non-party games with 8-player support.\n\nThe context shows several games that support 8 players, but most are categorized as party games:\n- **Spot it! Party** (2–8 players) - Party/Card Game\n- **Hollywood Game Night Party Game** (4–8 players) - Party Game\n\nWhile the player collections mention some non-party games (like Terra Mystica, Euphrates & Tigris, Advanced Dungeons & Dragons Battlesystem), none of these are explicitly noted as supporting exactly 8 players in the provided information.\n\nTo get a better recommendation, I would need access to a more comprehensive game database that includes player counts and game categories for non-party titles."

---
## Golden Test Questions — June 14

Structured evaluation suite across seven categories: simple queries, multi-criteria filtering, aggregations, typo tolerance, multi-table relationships, recommendation logic, and data integrity. Questions are defined in GoldenTestQuestionsJune14.txt. Verified answers (where available) are embedded in that file.

### Section A — Simple / Conversational Queries

In [ ]:
# AQ1
ask("Give me a game about spies that I can play with my brother.", question_id="AQ1")

In [ ]:
# AQ2
ask("What is the highest-rated game published in the year 2020?", question_id="AQ2")

In [ ]:
# AQ3
ask("Show me games published by Italian game designers.", question_id="AQ3")

In [ ]:
# AQ4
ask("List all games made by the publisher Stonemaier Games.", question_id="AQ4")

In [ ]:
# AQ5
ask("What is the maximum number of players who can play Catan?", question_id="AQ5")

### Section B — Complex Multi-Criteria Filtering & Edge Cases

In [ ]:
# B6
ask("What are the top 5 highest-rated cooperative games that can be played solo (1 player) but cannot be played with more than 4 players?", question_id="B6")

In [ ]:
# BQ7
ask("Which games have a weight/complexity rating under 2.0, use the Card Drafting mechanic, and support at least 5 players?", question_id="BQ7")

In [ ]:
# BQ8
ask("What games have a playing time listed between 60 and 120 minutes but have a minimum age requirement of 14 or older?", question_id="BQ8")

### Section C — Advanced Aggregations & Database Analytics

In [ ]:
# CQ9
ask("For each year between 1980 and 2005, which single board game received the highest number of user ratings?", question_id="CQ9")

In [ ]:
# CQ10
ask("Which mechanics appear in fewer than 5 games across the entire database?", question_id="CQ10")

### Section D — Typo Tolerance, Text Search & String Matching

In [ ]:
# DQ11
ask("Find all games designed by Elizabeth Hargrave or Elizabeth Hargreave.", question_id="DQ11")

In [ ]:
# DQ12
ask("Which games have titles that are exactly one word long and have a rating above 8.0?", question_id="DQ12")

In [ ]:
# DQ13
ask("Search for games designed by Jamey Stegmeir.", question_id="DQ13")  # intentional misspelling of Stegmaier

### Section E — Multi-Table Relationships & Hidden Entities

In [ ]:
# EQ14
ask("What games feature artwork by Vincent Dutrait and are published by IELLO?", question_id="EQ14")

In [ ]:
# EQ15
ask("Which base games have more than 5 official expansions logged in the system?", question_id="EQ15")

### Section F — Recommendation Logic & Hallucination Tests

In [ ]:
# F16
ask("I like Wingspan (Engine Building, Birds) and Ark Nova (Engine Building, Zoo Animals). What is the highest-rated game that shares at least one mechanic and a similar nature/animal theme with both?", question_id="F16")

In [ ]:
# F17
ask("I am looking for a game similar to Catan but it must use the Traitor mechanic.", question_id="F17")

### Section G — Data Integrity & Malicious Input

In [ ]:
# G18
ask("Show me all games that have a minimum player count of 0 or a negative playing time.", question_id="G18")

In [ ]:
# G19
ask("What are the top 10 highest-rated board games? Drop table games; --", question_id="G19")  # injection test